# 문서·파일 정보 통합 및 `-었-` 결합 성향 분류

기존 `0.4_document정보처리.ipynb`의 작업을 순서대로 정리한 통합판이다.

## 처리 흐름

1. 기본 문서 정보와 `0.3_document었결합률.ipynb`에서 만든 문서 결합 통계를 `docu_id`로 병합
2. 문서 로그오즈비를 5등급 및 `회피·중립·선호` 3성향으로 분류
3. 기본 파일 정보와 파일 결합 통계를 `file_id`로 병합
4. 파일 로그오즈비도 문서와 동일한 기준으로 분류
5. 파일 통계 컬럼에 `file_` 접두사를 붙여 각 문서에 병합
6. 모든 정보를 담은 문서 단위 통합 파일과 파일 단위 통합 파일 저장

문서 단위 통계에는 `docu_`, 파일 단위 통계에는 `file_` 접두사를 붙여 같은 이름의 컬럼이 충돌하지 않도록 한다.


In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# ---------- 입력 ----------
BASE_DOCU_CSV = Path(r"..\csv\original_csv\세종문어_document_정보.csv")
DOCU_STATS_CSV = Path(r"..\csv\original_csv\세종문어_document_었결합률.csv")
BASE_FILE_CSV = Path(r"..\csv\original_csv\세종문어_file_정보.csv")
FILE_STATS_CSV = Path(r"..\csv\original_csv\세종문어_file_정보2.csv")

# ---------- 출력 ----------
OUTPUT_DOCU_CSV = Path(r"..\csv\original_csv\세종문어_document_정보_통합.csv")
OUTPUT_FILE_CSV = Path(r"..\csv\original_csv\세종문어_file_정보_통합.csv")

# 기본 문서 정보에 이미 들어 있는 문서 통계도 docu_ 접두사로 정리한다.
BASE_DOCU_METRIC_COLS = [
    "sen_count", "sen_count_has_E", "sen_count_not_quote",
    "sen_count_has_E_not_quote", "base_count_not_quote",
    "dominant_EN_No", "dominant_EN_label", "dominant_count", "dominant_ratio",
]


In [ ]:
def read_unique_csv(path: Path, key: str, label: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{label} 파일이 없습니다: {path}")

    data = pd.read_csv(path, low_memory=False)
    if key not in data.columns:
        raise ValueError(f"{label}에 키 컬럼 '{key}'이 없습니다: {path}")

    data = data.dropna(subset=[key]).copy()
    data[key] = data[key].astype("string").str.strip()

    duplicated = data[key].duplicated(keep=False)
    if duplicated.any():
        examples = data.loc[duplicated, key].drop_duplicates().head(10).tolist()
        raise ValueError(f"{label}의 {key}가 중복됩니다. 예: {examples}")
    return data


def add_attachment_rate(data: pd.DataFrame) -> pd.DataFrame:
    """결합률이 없으면 결합 문장 수/분석 대상 문장 수로 생성한다."""
    data = data.copy()
    rate_col = "었_결합률"
    denominator_col = "body_not_quote_count"
    numerator_col = "body_not_quote_and_었_count"

    if rate_col not in data.columns:
        missing = {denominator_col, numerator_col} - set(data.columns)
        if missing:
            raise ValueError(f"결합률 계산에 필요한 컬럼이 없습니다: {sorted(missing)}")
        denominator = pd.to_numeric(data[denominator_col], errors="coerce").fillna(0)
        numerator = pd.to_numeric(data[numerator_col], errors="coerce").fillna(0)
        data[rate_col] = np.divide(
            numerator,
            denominator,
            out=np.zeros(len(data), dtype="float64"),
            where=denominator.ne(0),
        )
    return data


def classify_log_odds_5(value):
    """로그오즈비를 강한 회피(1)~강한 선호(5)로 분류한다."""
    if pd.isna(value):
        return pd.NA
    if value <= -1.5:
        return 1
    if value <= -0.5:
        return 2
    if value < 0.5:
        return 3
    if value < 1.5:
        return 4
    return 5


def add_attachment_classification(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    log_col = "었_결합_로그오즈비"
    if log_col not in data.columns:
        raise ValueError(f"분류에 필요한 컬럼이 없습니다: {log_col}")

    data["었_결합_등급"] = (
        data[log_col].apply(classify_log_odds_5).astype("Int64")
    )
    data["었_결합_성향"] = data["었_결합_등급"].map({
        1: "회피", 2: "회피", 3: "중립", 4: "선호", 5: "선호",
    })
    return data


def replace_overlapping_columns(
    base: pd.DataFrame,
    update: pd.DataFrame,
    key: str,
) -> pd.DataFrame:
    """기존의 같은 이름 컬럼은 제거하고 최신 update 컬럼으로 교체·병합한다."""
    update_cols = [column for column in update.columns if column != key]
    overlap = [column for column in update_cols if column in base.columns]
    base_clean = base.drop(columns=overlap)
    return base_clean.merge(update, on=key, how="left", validate="one_to_one")


In [ ]:
if not DOCU_STATS_CSV.exists():
    raise FileNotFoundError(
        f"문서 결합 통계가 없습니다: {DOCU_STATS_CSV}\n"
        "먼저 0.3_document었결합률.ipynb를 마지막 저장 셀까지 실행하세요."
    )

df_docu_base = read_unique_csv(BASE_DOCU_CSV, "docu_id", "기본 문서 정보")
df_docu_stats = read_unique_csv(DOCU_STATS_CSV, "docu_id", "문서 결합 통계")
df_file_base = read_unique_csv(BASE_FILE_CSV, "file_id", "기본 파일 정보")
df_file_stats = read_unique_csv(FILE_STATS_CSV, "file_id", "파일 결합 통계")

print(f"기본 문서 정보: {len(df_docu_base):,}행")
print(f"문서 결합 통계: {len(df_docu_stats):,}행")
print(f"기본 파일 정보: {len(df_file_base):,}행")
print(f"파일 결합 통계: {len(df_file_stats):,}행")


## 1. 문서 정보 통합·분류

기본문서정보와 문서별 결합 통계를 합하고 로그오즈비를 기준으로 문서를 분류한다.


In [ ]:
# 분류명 정리
if "category" in df_docu_base.columns:
    df_docu_base["category"] = df_docu_base["category"].replace({
        "보도·해설": "보도해설",
        "상상-일반": "허구일반",
        "상상-아동": "허구아동",
    })

df_docu_stats = add_attachment_rate(df_docu_stats)
df_docu_stats = add_attachment_classification(df_docu_stats)

required_docu_stats = {
    "docu_id", "body_not_quote_count", "body_not_quote_and_었_count",
    "었_결합률", "었_결합_오즈비", "었_결합_로그오즈비",
    "었_결합_등급", "었_결합_성향",
}
missing_docu_stats = sorted(required_docu_stats - set(df_docu_stats.columns))
if missing_docu_stats:
    raise ValueError(f"문서 결합 통계 컬럼이 부족합니다: {missing_docu_stats}")

df_docu_merged = replace_overlapping_columns(
    df_docu_base,
    df_docu_stats,
    key="docu_id",
)

# category와 매체를 함께 쓰는 분류도 생성
if {"category", "매체"}.issubset(df_docu_merged.columns):
    df_docu_merged["category2"] = (
        df_docu_merged["category"].astype("string").str.strip()
        + "_"
        + df_docu_merged["매체"].astype("string").str.strip()
    )

# 문서 단위 통계임을 명확히 하기 위해 docu_ 접두사 사용
docu_stats_cols = [column for column in df_docu_stats.columns if column != "docu_id"]
docu_metric_cols = [
    column for column in BASE_DOCU_METRIC_COLS + docu_stats_cols
    if column in df_docu_merged.columns
]
df_docu_merged = df_docu_merged.rename(
    columns={column: f"docu_{column}" for column in dict.fromkeys(docu_metric_cols)}
)

print(f"문서 정보 병합 완료: {len(df_docu_merged):,}행")
df_docu_merged[[
    "docu_id", "file_id", "category", "docu_었_결합률",
    "docu_었_결합_로그오즈비", "docu_었_결합_등급", "docu_었_결합_성향",
]].head()


## 2. 파일 정보 통합·분류

여기서 파일은 369개 말뭉치 파일 단위다. 한 파일 안에 여러 문서가 들어 있으므로 문서보다 값이 평균화되는 경향이 있다.


In [ ]:
df_file_stats = add_attachment_rate(df_file_stats)
df_file_stats = add_attachment_classification(df_file_stats)

required_file_stats = {
    "file_id", "body_not_quote_count", "body_not_quote_and_었_count",
    "었_결합률", "었_결합_오즈비", "었_결합_로그오즈비",
    "었_결합_등급", "었_결합_성향",
}
missing_file_stats = sorted(required_file_stats - set(df_file_stats.columns))
if missing_file_stats:
    raise ValueError(f"파일 결합 통계 컬럼이 부족합니다: {missing_file_stats}")

df_file_unified = replace_overlapping_columns(
    df_file_base,
    df_file_stats,
    key="file_id",
)

file_stats_cols = [column for column in df_file_stats.columns if column != "file_id"]
file_attach = df_file_unified[["file_id", *file_stats_cols]].rename(
    columns={column: f"file_{column}" for column in file_stats_cols}
)

print(f"파일 정보 병합 완료: {len(df_file_unified):,}행")
df_file_unified[[
    "file_id", "었_결합률", "었_결합_로그오즈비", "었_결합_등급", "었_결합_성향",
]].head()


## 3. 파일 정보를 각 문서에 붙여 최종 통합

각 문서 행에는 문서 자체의 `docu_...` 통계와 그 문서가 속한 파일의 `file_...` 통계가 함께 들어간다.


In [ ]:
if "file_id" not in df_docu_merged.columns:
    raise ValueError("문서 정보에 file_id가 없어 파일 정보를 병합할 수 없습니다.")

df_docu_merged["file_id"] = df_docu_merged["file_id"].astype("string").str.strip()
file_attach["file_id"] = file_attach["file_id"].astype("string").str.strip()

df_docu_unified = df_docu_merged.merge(
    file_attach,
    on="file_id",
    how="left",
    validate="many_to_one",
)

print(f"최종 문서 통합 완료: {len(df_docu_unified):,}행 × {len(df_docu_unified.columns):,}열")
df_docu_unified[[
    "docu_id", "file_id",
    "docu_었_결합률", "docu_었_결합_성향",
    "file_었_결합률", "file_었_결합_성향",
]].head()


In [ ]:
assert df_docu_unified["docu_id"].is_unique
assert df_file_unified["file_id"].is_unique
assert len(df_docu_unified) == len(df_docu_base)
assert len(df_file_unified) == len(df_file_base)

assert df_docu_unified["docu_었_결합률"].dropna().between(0, 1).all()
assert df_file_unified["었_결합률"].dropna().between(0, 1).all()

missing_docu_stats_count = df_docu_unified["docu_었_결합_로그오즈비"].isna().sum()
missing_file_stats_count = df_docu_unified["file_었_결합_로그오즈비"].isna().sum()

print(f"문서 통계 미병합: {missing_docu_stats_count:,}개")
print(f"파일 통계 미병합 문서: {missing_file_stats_count:,}개")
print("검증 완료")


In [ ]:
OUTPUT_DOCU_CSV.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE_CSV.parent.mkdir(parents=True, exist_ok=True)

df_docu_unified.to_csv(OUTPUT_DOCU_CSV, index=False, encoding="utf-8-sig")
df_file_unified.to_csv(OUTPUT_FILE_CSV, index=False, encoding="utf-8-sig")

print(f"문서 통합 파일 저장: {OUTPUT_DOCU_CSV}")
print(f"파일 통합 파일 저장: {OUTPUT_FILE_CSV}")
print(f"완료 시각: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
